<a href="https://colab.research.google.com/github/AmirthaaR/Network-Intrusion-Detection-using-PySpark/blob/main/Network_Intrusion_Detection_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("NetworkIntrusionDetection") \
    .getOrCreate()

print("PySpark version:", pyspark.__version__)
print("Spark session created successfully!")

PySpark version: 4.0.4
Spark session created successfully!


Download the dataset

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv to Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


Load and inspect the dataset

In [5]:
import glob
from pyspark.sql.functions import col, trim

csv_files = glob.glob("/content/*DDos*.csv")
print("Files found:", csv_files)

file_path = csv_files[0]

df = spark.read.csv(file_path, header=True, inferSchema=True)

df = df.toDF(*[c.strip() for c in df.columns])

print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

Files found: ['/content/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']
Rows: 225745
Columns: 79
root
 |-- Destination Port: integer (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |-- Total Length of Bwd Packets: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packets/s: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow 

Check the attack labels

In [6]:
label_col = next(c for c in df.columns if c.lower() == "label")

df.groupBy(label_col).count().orderBy(col("count").desc()).show(truncate=False)

+------+------+
|Label |count |
+------+------+
|DDoS  |128027|
|BENIGN|97718 |
+------+------+



Check missing and invalid values

In [9]:
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql.types import NumericType

missing_exprs = []

for field in df.schema.fields:
    c = field.name
    if isinstance(field.dataType, NumericType):
        missing_exprs.append(
            count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
        )
    else:
        missing_exprs.append(
            count(when(col(c).isNull(), c)).alias(c)
        )

missing_df = df.select(*missing_exprs)
missing_df.show(truncate=False)

+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+------------------+--------------------+------------

Clean column names and remove invalid rows

In [10]:
from pyspark.sql.functions import col, trim
from pyspark.sql.types import NumericType

df = df.toDF(*[
    c.strip().replace(" ", "_").replace("/", "_")
    for c in df.columns
])

label_col = "Label"

df = df.withColumn(label_col, trim(col(label_col)))

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, NumericType)
]

for c in numeric_cols:
    df = df.withColumn(
        c,
        when(col(c).isNull() | isnan(col(c)), None).otherwise(col(c))
    )

df = df.dropna(subset=["Label"])

print("Rows after basic cleaning:", df.count())
print("Columns:", len(df.columns))
df.groupBy("Label").count().show()

Rows after basic cleaning: 225745
Columns: 79
+------+------+
| Label| count|
+------+------+
|BENIGN| 97718|
|  DDoS|128027|
+------+------+



Check for infinite values

In [12]:
from pyspark.sql.functions import col, when, count, lit
from pyspark.sql.types import DoubleType, FloatType

float_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType))
]

infinite_exprs = [
    count(
        when(
            (col(c) == lit(float("inf"))) |
            (col(c) == lit(float("-inf"))),
            1
        )
    ).alias(c)
    for c in float_cols
]

if infinite_exprs:
    df.select(*infinite_exprs).show(truncate=False)
else:
    print("No floating-point columns found.")


+----------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+-----------+------------+-----------+-------------+-------------+------------------+-----------------+----------------------+-------------------+--------------------+--------------------+-----------+----------+---------+--------+
|Fwd_Packet_Length_Mean|Fwd_Packet_Length_Std|Bwd_Packet_Length_Mean|Bwd_Packet_Length_Std|Flow_Bytes_s|Flow_Packets_s|Flow_IAT_Mean|Flow_IAT_Std|Fwd_IAT_Mean|Fwd_IAT_Std|Bwd_IAT_Mean|Bwd_IAT_Std|Fwd_Packets_s|Bwd_Packets_s|Packet_Length_Mean|Packet_Length_Std|Packet_Length_Variance|Average_Packet_Size|Avg_Fwd_Segment_Size|Avg_Bwd_Segment_Size|Active_Mean|Active_Std|Idle_Mean|Idle_Std|
+----------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+-----------+------------+-----------+-------------+------------

Clean infinite values

In [13]:
from pyspark.sql.functions import col, lit, when

for c in float_cols:
    df = df.withColumn(
        c,
        when(
            (col(c) == lit(float("inf"))) |
            (col(c) == lit(float("-inf"))),
            None
        ).otherwise(col(c))
    )

print("Infinite values replaced with null.")


Infinite values replaced with null.


Handle missing values

In [14]:
from pyspark.sql.functions import coalesce

for c in numeric_cols:
    df = df.withColumn(c, coalesce(col(c), lit(0)))

print("Missing numeric values handled.")
print("Rows:", df.count())

Missing numeric values handled.
Rows: 225745


Prepare labels and features

In [15]:

from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler

label_col = "Label"

# Remove duplicate column names, if any
print("Total columns:", len(df.columns))
print("Target labels:")
df.groupBy(label_col).count().show()

# Convert text labels into numeric labels
label_indexer = StringIndexer(
    inputCol=label_col,
    outputCol="label",
    stringOrderType="alphabetAsc"
)

# Select numeric features, excluding the original and indexed labels
feature_cols = [
    c for c in numeric_cols
    if c in df.columns and c not in ["label"]
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print("Number of features:", len(feature_cols))


Total columns: 79
Target labels:
+------+------+
| Label| count|
+------+------+
|BENIGN| 97718|
|  DDoS|128027|
+------+------+

Number of features: 78


Split the dataset

In [16]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 180535
Testing rows: 45210


Build the pipeline and train the model

In [17]:

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler

label_indexer = StringIndexer(
    inputCol="Label",
    outputCol="label",
    stringOrderType="alphabetAsc"
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

pipeline = Pipeline(stages=[label_indexer, assembler, rf])

rf_model = pipeline.fit(train_df)

print("Random Forest training completed!")


Random Forest training completed!


Evaluate the model

In [18]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions = rf_model.transform(test_df)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.9998
F1 Score: 0.9998


Check DDoS detection performance

In [19]:

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col, sum as spark_sum, when

print("Actual labels:")
predictions.groupBy("Label").count().show()

print("Predictions by class:")
predictions.groupBy("Label", "prediction").count().orderBy("Label", "prediction").show()

# Confusion matrix
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Per-class precision and recall
for class_value, class_name in [(0.0, "BENIGN"), (1.0, "DDoS")]:
    tp = predictions.filter(
        (col("label") == class_value) & (col("prediction") == class_value)
    ).count()
    fp = predictions.filter(
        (col("label") != class_value) & (col("prediction") == class_value)
    ).count()
    fn = predictions.filter(
        (col("label") == class_value) & (col("prediction") != class_value)
    ).count()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    print(f"\n{class_name}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")


Actual labels:
+-----+-----+
|Label|count|
+-----+-----+
|  0.0|19530|
|  1.0|25680|
+-----+-----+

Predictions by class:
+-----+----------+-----+
|Label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|19530|
|  1.0|       0.0|   11|
|  1.0|       1.0|25669|
+-----+----------+-----+

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|19530|
|  1.0|       0.0|   11|
|  1.0|       1.0|25669|
+-----+----------+-----+


BENIGN
Precision: 0.9994
Recall:    1.0000
F1 Score:  0.9997

DDoS
Precision: 1.0000
Recall:    0.9996
F1 Score:  0.9998


Important validation before trusting the score

In [20]:
# Check whether the original label accidentally appears in features
print("Label column in feature list:", "Label" in feature_cols)
print("Indexed label in feature list:", "label" in feature_cols)
print("Feature count:", len(feature_cols))

# Confirm the target mapping learned by the pipeline
print("Label mapping:", rf_model.stages[0].labels)

Label column in feature list: False
Indexed label in feature list: False
Feature count: 78
Label mapping: ['BENIGN', 'DDoS']


Train Logistic Regression

In [21]:

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, VectorAssembler

label_indexer_lr = StringIndexer(
    inputCol="Label",
    outputCol="label",
    stringOrderType="alphabetAsc"
)

assembler_lr = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=50,
    regParam=0.01
)

lr_pipeline = Pipeline(stages=[
    label_indexer_lr,
    assembler_lr,
    lr
])

lr_model = lr_pipeline.fit(train_df)

print("Logistic Regression training completed!")


Logistic Regression training completed!


Evaluate Logistic Regression

In [22]:

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr_predictions = lr_model.transform(test_df)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )
    print(f"{metric}: {evaluator.evaluate(lr_predictions):.4f}")


accuracy: 0.9844
f1: 0.9844
weightedPrecision: 0.9848
weightedRecall: 0.9844


Check the Random Forest feature importance

In [23]:

import pandas as pd

rf_classifier = rf_model.stages[-1]

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_classifier.featureImportances.toArray()
})

importance_df = importance_df.sort_values(
    "Importance", ascending=False
)

print("Top 15 important features:")
print(importance_df.head(15).to_string(index=False))


Top 15 important features:
                    Feature  Importance
      Fwd_Packet_Length_Max    0.161010
Total_Length_of_Fwd_Packets    0.100930
       Avg_Fwd_Segment_Size    0.064776
     Fwd_Packet_Length_Mean    0.059795
     Init_Win_bytes_forward    0.050581
        Fwd_Header_Length34    0.050021
              Fwd_IAT_Total    0.044836
                Fwd_IAT_Max    0.044820
        Fwd_Header_Length55    0.044614
           act_data_pkt_fwd    0.042800
          Subflow_Fwd_Bytes    0.038782
                Fwd_IAT_Std    0.030268
          Total_Fwd_Packets    0.027035
           Destination_Port    0.025354
                Bwd_IAT_Max    0.021215


Strengthen the evaluation

In [24]:

from pyspark.sql.functions import rand

# Split each class separately to preserve class proportions
benign_df = df.filter(col("Label") == "BENIGN")
ddos_df = df.filter(col("Label") == "DDoS")

benign_train, benign_test = benign_df.randomSplit([0.8, 0.2], seed=123)
ddos_train, ddos_test = ddos_df.randomSplit([0.8, 0.2], seed=123)

train_df2 = benign_train.unionByName(ddos_train)
test_df2 = benign_test.unionByName(ddos_test)

print("Training rows:", train_df2.count())
print("Testing rows:", test_df2.count())

print("Training class distribution:")
train_df2.groupBy("Label").count().show()

print("Testing class distribution:")
test_df2.groupBy("Label").count().show()


Training rows: 180840
Testing rows: 44905
Training class distribution:
+------+------+
| Label| count|
+------+------+
|BENIGN| 78247|
|  DDoS|102593|
+------+------+

Testing class distribution:
+------+-----+
| Label|count|
+------+-----+
|BENIGN|19471|
|  DDoS|25434|
+------+-----+



Retrain Random Forest on the new split

In [25]:

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler

label_indexer2 = StringIndexer(
    inputCol="Label",
    outputCol="label",
    stringOrderType="alphabetAsc"
)

assembler2 = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

rf2 = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

pipeline2 = Pipeline(stages=[label_indexer2, assembler2, rf2])

rf_model2 = pipeline2.fit(train_df2)

print("Random Forest retrained on the new split!")


Random Forest retrained on the new split!


Evaluate the new model

In [26]:

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions2 = rf_model2.transform(test_df2)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )
    print(f"{metric}: {evaluator.evaluate(predictions2):.4f}")

print("Predictions by actual and predicted class:")
predictions2.groupBy("label", "prediction").count().orderBy(
    "label", "prediction"
).show()


accuracy: 0.9999
f1: 0.9999
weightedPrecision: 0.9999
weightedRecall: 0.9999
Predictions by actual and predicted class:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|19470|
|  0.0|       1.0|    1|
|  1.0|       0.0|    5|
|  1.0|       1.0|25429|
+-----+----------+-----+



In [27]:

from pyspark.sql.functions import count, countDistinct

# Count distinct feature vectors, excluding the target label
distinct_features = df.select(*feature_cols).distinct().count()
total_rows = df.count()

print("Total rows:", total_rows)
print("Distinct feature records:", distinct_features)
print("Repeated feature records:", total_rows - distinct_features)

# Check whether identical feature vectors have conflicting labels
conflicting_groups = (
    df.groupBy(*feature_cols)
      .agg(countDistinct("Label").alias("label_count"))
      .filter(col("label_count") > 1)
      .count()
)

print("Feature groups with conflicting labels:", conflicting_groups)


Total rows: 225745
Distinct feature records: 223112
Repeated feature records: 2633
Feature groups with conflicting labels: 0


In [28]:

df_dedup = df.dropDuplicates(feature_cols)

print("Original rows:", df.count())
print("Rows after deduplication:", df_dedup.count())
print("Removed duplicate rows:", df.count() - df_dedup.count())

print("Class distribution:")
df_dedup.groupBy("Label").count().show()


Original rows: 225745
Rows after deduplication: 223112
Removed duplicate rows: 2633
Class distribution:
+------+------+
| Label| count|
+------+------+
|BENIGN| 95096|
|  DDoS|128016|
+------+------+



In [29]:

benign_df3 = df_dedup.filter(col("Label") == "BENIGN")
ddos_df3 = df_dedup.filter(col("Label") == "DDoS")

benign_train3, benign_test3 = benign_df3.randomSplit([0.8, 0.2], seed=2026)
ddos_train3, ddos_test3 = ddos_df3.randomSplit([0.8, 0.2], seed=2026)

train_df3 = benign_train3.unionByName(ddos_train3)
test_df3 = benign_test3.unionByName(ddos_test3)

rf_model3 = pipeline2.fit(train_df3)
predictions3 = rf_model3.transform(test_df3)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )
    print(f"{metric}: {evaluator.evaluate(predictions3):.4f}")

predictions3.groupBy("label", "prediction").count().orderBy(
    "label", "prediction"
).show()


accuracy: 0.9998
f1: 0.9998
weightedPrecision: 0.9998
weightedRecall: 0.9998
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|18991|
|  0.0|       1.0|    1|
|  1.0|       0.0|    7|
|  1.0|       1.0|25570|
+-----+----------+-----+



In [30]:

print("PROJECT: Network Intrusion Detection using PySpark")
print("-" * 55)
print("Dataset: CIC-IDS2017 Friday Afternoon DDoS")
print("Original records:", df.count())
print("Original features:", len(feature_cols))
print("Deduplicated records:", df_dedup.count())
print("Random Forest accuracy:",
      round(MulticlassClassificationEvaluator(
          labelCol="label",
          predictionCol="prediction",
          metricName="accuracy"
      ).evaluate(predictions3), 4))
print("Random Forest F1:",
      round(MulticlassClassificationEvaluator(
          labelCol="label",
          predictionCol="prediction",
          metricName="f1"
      ).evaluate(predictions3), 4))
print("Baseline Logistic Regression accuracy: 0.9844")
print("Status: Training and evaluation completed")


PROJECT: Network Intrusion Detection using PySpark
-------------------------------------------------------
Dataset: CIC-IDS2017 Friday Afternoon DDoS
Original records: 225745
Original features: 78
Deduplicated records: 223112
Random Forest accuracy: 0.9998
Random Forest F1: 0.9998
Baseline Logistic Regression accuracy: 0.9844
Status: Training and evaluation completed
